# Notebook : Analyse de Fréquence (Congruent vs Incongruent)

Ce notebook guide l'analyse **en fréquence** des données Congruent vs Incongruent et l'extraction d'attributs **issus** de ces bandes.

### Remarque importante

La démarche de ce notebook consiste d'abord à explorer et tester le code **individuellement sur un seul sujet**.

Une fois les étapes validées, une **fonction finale** regroupe tous les blocs de code. Cette fonction permet d'appliquer l'ensemble du pipeline de traitement à **tous les sujets** de manière automatisée.

> **Attention :** Si vous avez expérimenté avec différents paramètres (par exemple, d'autres méthodes pour **l'extraction des bandes de fréquence**) et que vous souhaitez utiliser des valeurs autres que celles définies par défaut, **ou si vous avez ajouté d'autres blocs de code ou de nouvelles méthodes**, n'oubliez pas de **mettre à jour la fonction finale** en conséquence avant de l'exécuter sur l'ensemble des données.

Le notebook reprend la structure "Type 1 / Type 2 / Type 3" introduite dans `01_preprocessing_notebook.ipynb` :
- **Type 1 — prêt à exécuter** : cellules complètes, sans modification nécessaire.
- **Type 2 — à personnaliser** : blocs contenant des paramètres à ajuster (balises `A_COMPLETER`).
- **Type 3 — exploration libre** : propositions d'analyses supplémentaires, prêtes à modifier.

## Pour bien démarrer
- Assurez-vous d'avoir exécuté le pipeline de prétraitement pour générer les fichiers `*_clean_epo.fif`. **(Note : L'analyse en fréquence se fait presque toujours sur les époques).**
- Activez l'environnement Python du cours et installez les dépendances (`pip install -r requirements.txt`).

## Objectifs pédagogiques
1. Charger les **époques (epochs)** cw_cong/cw_incong déjà prétraitées.
2. Calculer et comparer la **Puissance Spectrale de Densité (PSD)** pour les conditions Congruent vs Incongruent.
3. Appliquer une **analyse Temps-Fréquence (TFR)** pour visualiser l'évolution de la puissance dans le temps (ex: ERD/ERS).
4. **Extraire des attributs (features)** basés sur la puissance moyenne dans des bandes spécifiques (ex: Alpha, Bêta) en vue du Machine Learning.

## 0. Préparation et configuration

Nous commençons par configurer l'environnement d'analyse (imports, chemins, sélection du sujet).


### Bloc Type 1 — Imports et options globales

Initialise les bibliothèques nécessaires, masque certains avertissements MNE et fixe le style de figures Matplotlib.


In [ ]:
# -----------------------------------------------------------------------------
# Imports principaux et configuration globale
# Chaque instruction est commentée pour rappeler son rôle.
# -----------------------------------------------------------------------------
import warnings  # contrôle de l'affichage des avertissements Python
from pathlib import Path  # manipulation portable des chemins
import csv  # lecture de fichiers tabulés (.tsv)
import json  # sauvegarde des résultats intermédiaires

import matplotlib.pyplot as plt  # production des figures
import mne  # bibliothèque cœur pour les analyses EEG/MEG
from mne.time_frequency import tfr_morlet  # analyse temps-fréquence
import numpy as np  # opérations numériques vectorisées
import mne_bids  # outils pour gérer la structure BIDS avec MNE
from mne_bids import BIDSPath  # construction de chemins compatibles BIDS

warnings.filterwarnings('ignore', category=RuntimeWarning)  # masque certains avertissements MNE
mne.set_log_level('INFO')  # verbosité modérée pour suivre les étapes clés
plt.rcParams['figure.figsize'] = (10, 5)  # taille par défaut des figures Matplotlib

print('Versions utilisées:')  # journalise les versions pour la reproductibilité
print(' - mne      ', mne.__version__)  # version de MNE
print(' - numpy    ', np.__version__)  # version de NumPy
print(' - mne_bids ', mne_bids.__version__)  # version de mne-bids


### Bloc Type 1 — Définir le dossier BIDS et les dérivés

Localise le dossier BIDS du dataset *text_reading* et configure les dérivés locaux :
- `derivatives/preproc` pour les fichiers prétraités (sortie du module 01).
- `derivatives/textsemantic-erp` pour les ERP groupés fournis avec le dataset.
- `derivatives/textsemantic-analysis` pour les résultats générés par ce notebook (features, exports CSV, etc.).



In [ ]:
# -----------------------------------------------------------------------------
# Localisation des données BIDS et des dérivés nécessaires
# -----------------------------------------------------------------------------
root_bids =Path('tasks/text_reading/bids')

deriv_preproc = root_bids / 'derivatives' / 'preproc'
deriv_preproc.mkdir(parents=True, exist_ok=True)

deriv_erp = root_bids / 'derivatives' / 'textsemantic-erp'
deriv_analysis = root_bids / 'derivatives' / 'textsemantic-analysis'
deriv_analysis.mkdir(parents=True, exist_ok=True)


print('Chemins vérifiés:')
print(' - BIDS root            :', root_bids.resolve())
print(' - dérivés préproc      :', deriv_preproc.resolve())
print(' - dérivés ERP (option) :', deriv_erp.resolve())
print(' - dérivés analyse      :', deriv_analysis.resolve())
print('EVENT_ID', EVENT_ID)



### Bloc Type 1 — Lister les participants disponibles

Lit `participants.tsv` (convention BIDS) pour récupérer les identifiants `sub-XX` disponibles. 
En Type 2 ci-dessous, nous choisirons l'un de ces sujets.


In [ ]:
# -----------------------------------------------------------------------------
# Lecture de participants.tsv afin d'obtenir la liste des sujets présents
# -----------------------------------------------------------------------------
participants_tsv = root_bids / 'participants.tsv'  # chemin vers le fichier BIDS
subjects = []  # contiendra les identifiants sans le préfixe 'sub-'

with participants_tsv.open('r', encoding='utf-8') as f:  # ouverture du fichier en lecture
    reader = csv.reader(f, delimiter='	')  # lecture tabulée
    header = next(reader, None)  # saute l'entête (participant_id, ...)
    for row in reader:  # boucle sur chaque ligne restante
        if not row:  # ignore les lignes vides
            continue
        participant_id = row[0]  # première colonne = identifiant sujet
        if participant_id.startswith('sub-'):  # vérifie le format BIDS
            subjects.append(participant_id.replace('sub-', ''))  # stocke l'identifiant sans préfixe

print(f"Participants détectés ({len(subjects)}): {subjects}")  # affiche la liste obtenue


### Bloc Type 2 — Sélectionner un participant et une session

Modifiez les identifiants ci-dessous pour analyser un autre sujet. Chaque sujet dispose d'un unique run (`run = '01'`).



In [ ]:
# -----------------------------------------------------------------------------
# Choix du sujet/session/run à analyser (modifiable)
# -----------------------------------------------------------------------------
subject = '03'  # <--- remplacez par ex. '05' pour explorer un autre participant
session = '001'  # unique session disponible pour ce dataset
run = '01'  # un seul run textsemantic est fourni

print(f'Sujet en cours: sub-{subject}, session {session}, run {run}')



## 1. Charger un enregistrement prétraité

Nous chargeons le fichier `*_clean.fif` issu du pipeline précédent, appliquons un montage standard et vérifions les métadonnées clés.


### Bloc Type 1 — Fonction utilitaire de chargement

`load_processed_raw` centralise la construction du chemin de fichier, le chargement `Raw` MNE et l'application d'un montage 10-20 international.


In [ ]:
# -----------------------------------------------------------------------------
# Fonction utilitaire : chargement d'un fichier prétraité pour un sujet donné
# -----------------------------------------------------------------------------
def load_processed_raw(subject: str, session: str = '001', run: str = '01') -> mne.io.BaseRaw:
    # Construit le chemin BIDS du fichier *_processed.fif dans derivatives/preproc
    processed_bids = BIDSPath(
        root=deriv_preproc,
        subject=subject,
        session=session,
        task='textsemantic',
        run=run,
        datatype='eeg',
        suffix='eeg',
        processing='clean',
        extension='.fif'
    )
    fname = processed_bids.fpath
    if not fname.exists():  # garde-fou si le fichier manque
        raise FileNotFoundError(f'Fichier introuvable: {fname}')  # message explicite
    raw_obj = mne.io.read_raw_fif(fname, preload=True)  # charge en mémoire pour un accès rapide
    raw_obj.set_montage('standard_1020', match_case=False, on_missing='warn')  # assure la co-registration EEG
    return raw_obj  # renvoie l'objet Raw prêt à l'emploi

raw = load_processed_raw(subject, session=session, run=run)  # chargement effectif
print(raw)  # résumé de l'objet Raw


### Bloc Type 1 — Extraire les événements textsemantic

Nous mappons les annotations BIDS (`Stimulus/S 21`, `Stimulus/S 22`) vers les labels `Congruent` / `Incongruent` via `ANNOTATION_MAP`. Ce bloc vérifie la disponibilité de chaque condition cible.



### Bloc Type 1 — Extraire les événements Cong/Incong depuis les annotations

La table `annotation_map` relie les identifiants bruts à des labels plus explicites.


In [ ]:
# -----------------------------------------------------------------------------
# Extraction des événements et renommage en labels lisibles
# -----------------------------------------------------------------------------
annotation_map = {
    'Stimulus/S 21': 'cw_cong',
    'Stimulus/S 22': 'cw_incong',
}

events, event_id = mne.events_from_annotations(raw)
selected_event_id = {}
for original, label in annotation_map.items():
    if np.str_(label) in event_id:
        selected_event_id[label] = event_id[np.str_(label)]

print('Étiquettes retenues:')
for label, code in selected_event_id.items():
    n_trials = int((events[:, 2] == code).sum())
    print(f' - {label:12s} → code {code:3d}, essais = {n_trials}')

focus_event_id = {label: code for label, code in selected_event_id.items() if label in FOCUS_CONDITIONS}

### Bloc Type 1 — Paramètres d'epoching

Définit la fenêtre temporelle autour des événements et calcule les epochs EEG associés.


In [ ]:
# -----------------------------------------------------------------------------
# Paramètres d'epoching et construction des epochs MNE
# -----------------------------------------------------------------------------

# Définition de la fenêtre temporelle pour chaque epoch :
tmin, tmax = -0.2, 0.8  # -200 ms avant l'événement (t=0) et +800 ms après.

# Définition de la période de "ligne de base" (baseline) :
baseline = (-0.2, 0.0)  # Utilise l'intervalle [-200 ms, 0 ms] (pré-stimulus)
                        # La moyenne de cette période sera soustraite de toute l'epoch.

# Création de l'objet Epochs
epochs = mne.Epochs(
    raw,  # Le signal continu (filtré) à découper
    events,  #  La matrice des événements modifiée
    event_id=selected_event_id,  # UTILISATION CORRECTE : Le dictionnaire propre
    tmin=tmin,  # Début de la fenêtre
    tmax=tmax,  # Fin de la fenêtre
    baseline=baseline,  # Période de correction de la ligne de base
    picks='eeg',  # Sélectionne UNIQUEMENT les canaux de type 'eeg'
    preload=True,  # Charge toutes les données des epochs en mémoire (RAM).
                  # REQUIS pour AutoReject, ICA, etc.
    detrend=None,  # N'applique pas de "detrending"
)

print(epochs)  # Affiche un résumé (nombre d'epochs, canaux, temps)

# Comptage final des essais pour chaque condition
# (en utilisant les étiquettes de 'selected_event_id')
trial_counts = {cond: len(epochs[cond]) for cond in epochs.event_id}

print('Essais conservés par condition :', trial_counts)  # Affichage console

## 3. Analyse Exploratoire en Fréquence (TFR)

Explorez les densités de puissance pour les essais Congruent vs Incongruent et observez les signatures spectrales autour de l'effet N400.



### Bloc Type 2 — Visualiser les spectres par condition

Tracez les TFRs pour les conditions Congruent/Incongruent sur la ligne médiane fronto-central (Fz, Cz).



In [ ]:
CHANNEL_CLUSTERS = {
    "midline_fc": ["Fz", "Cz"],
    "centro_parietal": ["Cz", "Pz", "P3", "P4"],
    "parietal": ["Pz", "P3", "P4"],
    "occipital": ["O1", "O2"],
}
# Paramètres pour le plot TFR
plot_freqs = np.arange(4.0, 31.0, 1.0)  # vecteur de fréquences de 4 à 30 Hz (pas 1 Hz)
plot_cycles = np.maximum(plot_freqs / 2.0, 2.5)  # nombre de cycles par fréquence (min 2 cycles)
picks = CHANNEL_CLUSTERS['midline_fc']  # liste des canaux à utiliser pour le plot TFR
plot_conditions = ['cw_cong', 'cw_incong'] # conditions à tracer
tfr_cache = {}  # dictionnaire vide pour mettre en cache les objets TFR calculés
TFR_BASELINE = (-0.2, 0.0)  # fenêtre de baseline en secondes pour la normalisation TFR
TFR_MODE = 'logratio'  # mode de baseline (ici logratio, équivalent dB / ratio logarithmique)
TFR_DECIM = 2  # facteur de décimation pour accélérer le calcul et réduire la résolution temporelle


In [ ]:
for label in plot_conditions:  # itère sur chaque condition à tracer (ici cw_cong correct et cw_incong correct)
    if label not in epochs.event_id or len(epochs[label]) == 0:  # vérifie que la condition existe et contient des epochs
        print(f'[skip] condition {label} manquante ou vide.')  # informe si la condition est absente ou vide
        continue  # passe à la condition suivante si non disponible

    print(f"Calcul TFR pour {label}...")  
    tfr_avg = tfr_morlet( 
        epochs[label].copy(),  # copie des epochs de la condition pour éviter de modifier l'objet original
        freqs=plot_freqs,  # vecteur des fréquences à analyser (ex: 4-30 Hz)
        n_cycles=plot_cycles,  # nombre de cycles pour chaque fréquence (contrôle résolution temps/fréq)
        picks=picks,  # sélection des canaux à utiliser (ex: Fz, Cz)
        average=True, # Calculer la moyenne des essais  # calcule la TFR moyenne sur les epochs (AverageTFR)
        return_itc=False,  # ne pas calculer la cohérence inter-trials (ITC)
        use_fft=True,  # utilise la FFT pour accélérer le calcul
        decim=TFR_DECIM,  # décimation temporelle pour réduire la résolution et accélérer
        n_jobs=-1, # Utiliser tous les coeurs  # parallélise sur tous les coeurs disponibles
        verbose=False,  # supprime les messages verbeux de tfr_morlet
    )
    tfr_avg.apply_baseline(TFR_BASELINE, mode=TFR_MODE, verbose=False)  # applique la normalisation baseline (p.ex. logratio)
    tfr_cache[label] = tfr_avg  # met en cache l'objet TFR calculé pour cette condition
    
    # Afficher le TFR (heatmap temps-fréquence)
    tfr_avg.plot(  # trace la TFR moyenne sous forme de heatmap
        picks=picks,  # canaux à afficher dans le plot
        title=f"TFR — {label} ({', '.join(picks)})",  # titre décrivant la condition et les canaux
        show=True,  # affiche la figure immédiatement
    )

# --- Plot de la Différence (Congruent - Incongruent) ---
if set(plot_conditions).issubset(tfr_cache):  # vérifie que les deux conditions sont présentes dans le cache
    print("Affichage de la différence TFR (Congruent - Incongruent)...")  # log avant d'afficher la différence
    diff_tfr = tfr_cache[plot_conditions[1]].copy() # Incongruent  # copie la TFR Incongruent comme base pour la différence
    diff_tfr.data -= tfr_cache[plot_conditions[0]].data # - Congruent  # soustrait la TFR Congruent pour obtenir la différence
    
    diff_tfr.plot(  # trace la TFR de différence
        picks=diff_tfr.ch_names,  # affiche tous les canaux présents dans l'objet diff_tfr
        title='Différence TFR : Congruent - Incongruent (midline)',  # titre du graphique de différence
        show=True,  # affiche la figure,
    )
else:
    print('Pas assez de conditions pour tracer la différence TFR.')  # message si la différence ne peut pas être calculée


### Bloc Type 3 — Visualiser les spectres pour d'autres clusters

Changez `picks` pour afficher d'autres régions (centro-pariétal, occipital) afin de localiser les effets ERP dans l'espace.



### Bloc Type 2 — Visualiser les topographies de fréquence (optionnel)

Utilisez `diff_tfr` pour explorer la distribution spatiale de la différence Incongruent - Congruent dans plusieurs bandes.



### Bloc Type 2 — Visualiser les spectres par condition

Inspectez les densités de puissance pour les conditions Congruent/Incongruent avant d'extraire les features ERP.



In [ ]:
# Nous réutilisons le 'diff_tfr' (Incongruent - Congruent) calculé ci-dessus
print("Affichage des topographies de la différence...")
    
# Définir les bandes d'intérêt
bands = {
    'Theta (4-7 Hz)': (4.0, 7.0),
    'Alpha (8-12 Hz)': (8.0, 12.0),
    'Beta (15-25 Hz)': (15.0, 25.0)
}

# Fenêtre de temps d'intérêt (post-stimulus)
tmin_topo, tmax_topo = 0.1, 0.5

fig, axes = plt.subplots(1, len(bands), figsize=(15, 5), constrained_layout=True)
if len(bands) == 1: axes = [axes] # Assurer que 'axes' est itérable

for ax, (band_name, (fmin, fmax)) in zip(axes, bands.items()):
    # Calculer la moyenne sur le temps et la fréquence
    diff_tfr.plot_topomap(
        fmin=fmin, fmax=fmax,
        tmin=tmin_topo, tmax=tmax_topo,
        axes=ax,
        show=False, 
        colorbar=True,
    )
plt.show()

### Bloc Type 3 — Explorer d'autres paramètres de bande

* Modifier `bands` pour ajouter d'autres bandes (gamma, delta) ou d'autres clusters.
* Ajuster `n_freqs`, `min_cycles` ou `decim` pour équilibrer résolution fréquentielle et bruit.
* Tester des configurations de `tmin/tmax` différentes pour cibler les effets post-stimulus.
* Utiliser `n_freqs` et `CHANNEL_CLUSTERS` pour ajouter les plots d'autres clusters.
* Utiliser `tfr_cache` pour ajouter les topomap de `cw_cong` où `cw_incong`.



## 4.  Extraction des attributs (features) de fréquence en (Evoked)

Pour l'analyse statistique traditionnelle, nous extrayons les features
à partir des objets Evoked (moyenne des essais par condition).
Chaque ligne du CSV final représentera une condition pour ce sujet.

### Bloc Type 1 — Configurer les bandes de fréquence et les utilitaires (Evoked)

In [ ]:
# -----------------------------------------------------------------------------
# Définitions ERP et fonctions utilitaires
# -----------------------------------------------------------------------------
FREQUENCY_BAND_WINDOWS = [
    {"name": "theta_midline_fc", "cluster": "midline_fc", "freq_range": (5.0, 7.0), "n_freqs": 3, "tmin": 0.200, "tmax": 0.600},
    {"name": "alpha_parietal", "cluster": "parietal", "freq_range": (8.0, 13.0), "n_freqs": 6, "tmin": 0.200, "tmax": 0.600},
    {"name": "beta_parietal", "cluster": "parietal", "freq_range": (13.0, 20.0), "n_freqs": 8, "tmin": 0.200, "tmax": 0.600},
]
BAND_NAMES = [config['name'] for config in FREQUENCY_BAND_WINDOWS]


def _resolve_picks(inst: mne.Evoked | mne.Epochs, picks: list) -> list:
    "Vérifie que les canaux existent dans les données."
    if isinstance(picks, str):
        picks = [picks]
    return [ch for ch in picks if ch in inst.ch_names]

def _safe_cycles_per_freq(
    freqs: np.ndarray,
    epoch_len: float,
    min_cycles: float,
    max_cycles: float,
    safe_fraction: float = 0.5
) -> np.ndarray:
    """
    Calcule n_cycles pour que le support temporel (n_cycles / f) 
    reste <= safe_fraction * epoch_len.
    Inspiré par votre nouvelle fonction.
    """
    max_allowed = safe_fraction * epoch_len * freqs
    n_cycles = np.minimum(max_cycles, np.maximum(min_cycles, max_allowed))
    return n_cycles

In [ ]:
def compute_band_power_evoked(
    evoked: mne.Evoked,
    picks: list,
    freq_range: tuple,
    tmin: float,
    tmax: float,
    n_freqs: int = 5,
    min_cycles: float = 3.0,
    max_cycles: float = 7.0,
    baseline: tuple | None = TFR_BASELINE,
    mode: str = TFR_MODE,
    decim: int = TFR_DECIM,
) -> float:
    """Compute the mean band power for a single Evoked object.

    Parameters
    ----------
    evoked : mne.Evoked
        Evoked (ERP) object containing averaged data across trials.
    picks : list
        List of channel names to include (cluster). Channels missing from `evoked`
        will be ignored.
    freq_range : tuple
        Frequency band as (fmin, fmax).
    tmin, tmax : float
        Time window (seconds) relative to the evoked times over which to average.
    n_freqs : int
        Number of frequencies to sample between fmin and fmax.
    min_cycles, max_cycles : float
        Min/max cycles used to build the Morlet wavelets (controls time/freq trade-off).
    baseline : tuple or None
        Baseline window for TFR normalization (None to skip).
    mode : str
        Baseline mode passed to `apply_baseline` (e.g. 'logratio').
    decim : int
        Temporal decimation factor for the TFR computation.

    Returns
    -------
    float
        Mean power averaged across channels, frequencies and times for the specified window.
        Returns np.nan if no requested channels are available.
    """
    # Keep only channels that actually exist in the Evoked object
    valid_picks = _resolve_picks(evoked, picks)
    if not valid_picks:
        # No matching channels -> cannot compute power for this cluster
        return np.nan

    # Build the frequency vector to analyze
    freqs = np.linspace(freq_range[0], freq_range[1], n_freqs)

    # Estimate epoch length (duration) from the evoked times array
    epoch_len = float(evoked.times[-1] - evoked.times[0])

    # Choose a safe number of cycles per frequency so that wavelet support
    # does not exceed a fraction of the epoch length (see helper)
    n_cycles = _safe_cycles_per_freq(freqs, epoch_len, min_cycles, max_cycles)

    # Compute the TFR using Morlet wavelets on the Evoked object.
    # For Evoked objects, averaging is performed internally (average=True by default).
    power = tfr_morlet(
        evoked.copy(),             # operate on a copy to avoid in-place changes
        freqs=freqs,               # frequencies to evaluate
        n_cycles=n_cycles,         # cycles per frequency (array)
        picks=valid_picks,         # channels to include
        return_itc=False,          # we only need power, not ITC
        use_fft=True,              # use FFT-based convolution for speed
        decim=decim,               # temporal decimation to speed up computation
        n_jobs=-1,                 # parallelize across available cores
        verbose=False,
    )

    # Apply baseline normalization if requested (e.g. logratio)
    if baseline is not None:
        power.apply_baseline(baseline, mode=mode, verbose=False)

    # Restrict the TFR to the requested time window before averaging
    power.crop(tmin, tmax)

    # power.data shape: (n_channels, n_freqs, n_times)
    # Return the grand mean across channels, frequencies and times as a scalar float
    return float(power.data.mean(axis=(0, 1, 2)))


In [ ]:
# --- 1. Calculer les Evokeds (ERPs) ---
print("Calcul des Evokeds (moyennes)...")
# Ajouter les conditions groupées
evokeds = {}
evokeds['cw_cong'] = epochs['cw_cong'].average()
evokeds['cw_incong'] = epochs['cw_incong'].average()

In [ ]:
# Construire une ligne de features par condition et calculer les puissances de bande (avec affichage de debug).
for condition, evk in evokeds.items():
    # Initialiser la ligne de features pour cette condition (une ligne par Evoked/condition)
    row_features = {
        'subject': subject,
        'session': session,
        'run': run,
        'condition': condition
    }

    # Afficher un en-tête court pour cette condition (evoked.nave = nombre d'epochs utilisés)
    print(f"\nTraitement de la condition : {condition}")

    for config in FREQUENCY_BAND_WINDOWS:
        # Résoudre les canaux (picks) pour cette fenêtre de fréquence (cluster)
        picks = CHANNEL_CLUSTERS.get(config['cluster'], [])
        picks_str = ','.join(picks) if picks else '[]'

        # on calcule la puissance de bande sur l'Evoked en utilisant des ondelettes Morlet
        # avec un choix sécurisé du nombre de cycles, baseline et décimation selon la config.
        power = compute_band_power_evoked(
            evk,
            picks,
            config['freq_range'],
            config['tmin'],
            config['tmax'],
            n_freqs=config.get('n_freqs', 5),
            min_cycles=config.get('min_cycles', 3.0),
            max_cycles=config.get('max_cycles', 7.0),
            baseline=config.get('baseline', TFR_BASELINE),
            mode=config.get('mode', TFR_MODE),
            decim=config.get('decim', TFR_DECIM),
        )

        # Affichage debug après chaque calcul
        if isinstance(power, float) and np.isnan(power):
            print(f"  {config['name']:20s} | picks={picks_str:10s} -> puissance=nan (aucun canal valide)")
        else:
            print(f"  {config['name']:20s} | picks={picks_str:10s} -> puissance={power:.6f}")

        # Stocker le résultat dans la ligne
        row_features[config['name']] = power

    # Impression récapitulative de la ligne complétée (aperçu)
    print("  -> Ligne de features complétée (aperçu) :",
          {k: row_features[k] for k in ['condition'] + BAND_NAMES})

### Bloc Type 3 — Explorer d'avantages d'autres options. 

* Modifier `FREQUENCY_BAND_WINDOWS` et `CHANNEL_CLUSTERS` pour ajouter d'autres bandes (gamma, delta) ou d'autres clusters/configurations.
* Ajuster `n_freqs`, `min_cycles` ou `decim` pour équilibrer résolution fréquentielle et bruit.
* Tester des configurations de `tmin/tmax` différentes pour cibler les effets post-stimulus.

## 5. Extraction des attributs (features) de fréquence à partir de (Epochs)

Comme altervative, nous extrayons les features à partir des objets Epochs (essais individuels).
Chaque ligne du CSV final représentera un essai.


In [ ]:
# -----------------------------------------------------------------------------
# Définitions spatiales et fréquentielles
# (Les configurations sont les mêmes que pour Evoked)
# -----------------------------------------------------------------------------
# FREQUENCY_BAND_WINDOWS et CHANNEL_CLUSTERS sont déjà définis

# --- Fonctions d'aide (Helpers) ---
# _resolve_picks et _safe_cycles_per_freq sont déjà définis

# On définit une nouvelle fonction pour les epochs
def compute_band_power_epochs(
    epochs: mne.Epochs,
    picks: list,
    freq_range: tuple,
    tmin: float,
    tmax: float,
    n_freqs: int = 5,
    min_cycles: float = 3.0,
    max_cycles: float = 7.0,
    baseline: tuple | None = TFR_BASELINE,
    mode: str = TFR_MODE,
    decim: int = TFR_DECIM,
) -> np.ndarray:
    """
    Calcule la puissance moyenne pour une bande de fréquence / fenêtre temporelle /
    cluster de canaux pour chaque essai des `epochs`.

    Retour
    ------
    np.ndarray (n_epochs,)
        Valeur moyenne de puissance pour chaque essai. Si aucun canal valide n'est trouvé
        pour le cluster demandé, renvoie un tableau rempli de NaN de longueur n_epochs.

    Principe
    --------
    - On construit un vecteur de fréquences (linspace entre fmin et fmax).
    - On calcule un nombre de cycles "sécurisé" par fréquence pour que le support temporel
      de l'ondelette ne dépasse pas une fraction raisonnable de la durée d'epoch.
    - On applique `tfr_morlet` avec average=False pour conserver la dimension par essai.
    - On applique la baseline (optionnelle), coupe la fenêtre temporelle d'intérêt,
      puis on moyenne sur canaux / fréquences / temps pour obtenir une valeur par essai.
    """
    # Vérifier et résoudre les canaux demandés : ne garder que ceux présents dans epochs
    valid_picks = _resolve_picks(epochs, picks)
    if not valid_picks:
        # Aucun canal valide : renvoyer un vecteur NaN pour chaque essai
        return np.full(len(epochs), np.nan)

    # Construire la grille fréquentielle (ex : 4 valeurs entre fmin et fmax)
    freqs = np.linspace(freq_range[0], freq_range[1], n_freqs)

    # Estimer la durée effective d'une epoch (en secondes) à partir des times
    epoch_len = float(epochs.times[-1] - epochs.times[0])

    # Calculer un nombre de cycles par fréquence "sécurisé" (évite des wavelets trop étalées)
    n_cycles = _safe_cycles_per_freq(freqs, epoch_len, min_cycles, max_cycles)

    # Calculer la TFR par essai (average=False) -> power.data shape: (n_epochs, n_channels, n_freqs, n_times)
    power = tfr_morlet(
        epochs.copy(),       # travail sur une copie pour ne pas modifier l'objet d'origine
        freqs=freqs,
        n_cycles=n_cycles,
        picks=valid_picks,
        average=False,       # conserver la dimension par essai
        return_itc=False,
        use_fft=True,
        decim=decim,
        n_jobs=-1,
        verbose=False,
    )

    # Appliquer la baseline si demandée (ex: 'logratio' ou 'zscore')
    if baseline is not None:
        power.apply_baseline(baseline, mode=mode, verbose=False)

    # Restreindre la TFR à l'intervalle temporel d'intérêt avant d'agréger
    power.crop(tmin, tmax)

    # Moyennage sur axes (channels, freqs, times) pour obtenir une valeur par essai
    # power.data est de forme (n_epochs, n_channels, n_freqs, n_times)
    # Résultat final : tableau (n_epochs,)
    return power.data.mean(axis=(1, 2, 3))

### Bloc Type 1 — Calculer les attributs issues de bande de fréquences avec les objects Epoched.

In [ ]:
import pandas as pd  # pour la manipulation de tableaux de données
# Nombre d'essais dans l'objet epochs
n_trials = len(epochs)
# On crée un mapping code -> label pour obtenir le nom lisible de la condition
code_to_label = {code: label for label, code in epochs.event_id.items()}
# Récupère la condition pour chaque événement (événements ordonnés)
conditions = [code_to_label.get(code, str(code)) for code in epochs.events[:, 2]]

# Prépare le dictionnaire d'information de base (index, condition, méta)
info = {
    'trial_index': np.arange(n_trials),
    'condition': conditions,
}
# Ajoute les méta (subject/session/run) si présents
info['subject'] = [subject] * n_trials

# Convertit en DataFrame vide (colonnes de base)
df_features = pd.DataFrame(info)

# Pour chaque fenêtre de bande, calcule la puissance par essai et l'ajoute au DataFrame
for config in FREQUENCY_BAND_WINDOWS:
    print(f"  ... calcul de {config['name']} (epochs)")
    power = compute_band_power_epochs(
        epochs,
        CHANNEL_CLUSTERS.get(config['cluster'], []),  # picks = canaux du cluster (ou [] si absent)
        config['freq_range'],  # intervalle fréquentiel (fmin, fmax)
        config['tmin'],        # début de la fenêtre temporelle d'intérêt
        config['tmax'],        # fin de la fenêtre temporelle d'intérêt
        n_freqs=config.get('n_freqs', 5),           # nb de fréquences à échantillonner
        min_cycles=config.get('min_cycles', 3.0),   # cycles min pour les ondelettes
        max_cycles=config.get('max_cycles', 7.0),   # cycles max pour les ondelettes
        baseline=config.get('baseline', TFR_BASELINE), # fenêtre de baseline pour normalisation
        mode=config.get('mode', TFR_MODE),            # mode de baseline (ex: 'logratio')
        decim=config.get('decim', TFR_DECIM),         # décimation temporelle
    )
    # Ajoute la colonne de puissance (une valeur par essai) au tableau
    df_features[config['name']] = power

df_features.head()  # affiche un aperçu des premières lignes du tableau final

### Bloc Type 3 — Explorer d'avantages d'autres options. 

* Modifier `FREQUENCY_BAND_WINDOWS` et `CHANNEL_CLUSTERS` pour ajouter d'autres bandes (gamma, delta) ou d'autres clusters/configurations.
* Ajuster `n_freqs`, `min_cycles` ou `decim` pour équilibrer résolution fréquentielle et bruit.
* Tester des configurations de `tmin/tmax` différentes pour cibler les effets post-stimulus.

## 5. Factoriser le pipeline pour tous les sujets

Objectifs : créer des fonctions modulaires qui calcule les attributs automatiquements pour tous les sujets. 

In [ ]:
def process_subject_features(subject, session, run, configs, t_epoch):
    """
    Exécute le pipeline complet d'extraction d'attributs (features)
    issue de bande de fréquence pour un seul sujet.
    
    Retourne 3 DataFrames: 
        1. df_evoked_features_freq (1 ligne par condition)
        2. df_epoch_features_freq (1 ligne par essai)
        3. df_behavior (1 ligne par essai)
    
    Args:
        subject (str): ID du sujet (ex: '01').
        session (str): ID de la session (ex: '001').
        run (str): ID du run (ex: '01').
        configs (dict): Dictionnaire contenant les configurations 
                        (CHANNEL_CLUSTERS, FREQUENCY_BAND_WINDOWS).
        t_epoch (dict): Dictionnaire des temps d'epoching (tmin, tmax, baseline).

    Returns:
        tuple: (df_evoked_features, df_epoch_features, df_behavior)
    """
    print(f"\n--- Traitement Sujet : {subject} ---")
    
    # --- 0. Décompression des configurations ---
    CHANNEL_CLUSTERS = configs['CHANNEL_CLUSTERS']
    FREQUENCY_BAND_WINDOWS = configs['FREQUENCY_BAND_WINDOWS'] # CORRECTION

    # --- 1. Chargement des données ---
    raw = load_processed_raw(subject, session=session, run=run)

    # --- 2. Parsing des événements (Logique "Look-Ahead") ---
    print("  ... 1/6 Parsing des événements...")
    annotation_map = {
        'Stimulus/S 21': 'cw_cong',
        'Stimulus/S 22': 'cw_incong',
    }

    events, event_id = mne.events_from_annotations(raw)
    selected_event_id = {}
    for original, label in annotation_map.items():
        if np.str_(label) in event_id:
            selected_event_id[label] = event_id[np.str_(label)]

    print('Étiquettes retenues:')
    for label, code in selected_event_id.items():
        n_trials = int((events[:, 2] == code).sum())
        print(f' - {label:12s} → code {code:3d}, essais = {n_trials}')

    focus_event_id = {label: code for label, code in selected_event_id.items() if label in FOCUS_CONDITIONS}
    # --- 3. Création des Époques ---
    print("  ... 2/6 Création des époques...")
    epochs = mne.Epochs(raw, events, event_id=selected_event_id,
                        tmin=t_epoch['tmin'], tmax=t_epoch['tmax'],
                        baseline=t_epoch['baseline'], picks='eeg',
                        preload=True, detrend=None)

    # --- 4. Calcul des Evokeds (ERPs) ---
    print("  ... 3/6 Calcul des ERPs (Evokeds)...")
    evokeds = {}
    evokeds['cw_cong'] = epochs['cw_cong'].average()
    evokeds['cw_incong'] = epochs['cw_incong'].average()

    # --- 5. Calcul des features (Evokeds) ---
    print("  ... 4/6 Extraction des features (Evokeds)...")
    evoked_feature_rows = []
    for condition, evk in evokeds.items():
         # Initialiser la ligne de features pour cette condition
        row_features = {'subject': subject, 'session': session, 'run': run, 'condition': condition}
        for config in FREQUENCY_BAND_WINDOWS:
            power = compute_band_power_evoked(
                evk, 
                CHANNEL_CLUSTERS.get(config['cluster'], []),
                config['freq_range'], config['tmin'], config['tmax'],
                n_freqs=config.get('n_freqs', 5),
                min_cycles=config.get('min_cycles', 3.0),
                max_cycles=config.get('max_cycles', 7.0),
                baseline=config.get('baseline', TFR_BASELINE),
                mode=config.get('mode', TFR_MODE),
                decim=config.get('decim', TFR_DECIM),
            )
            row_features[config['name']] = power
        evoked_feature_rows.append(row_features)
    df_evoked_features = pd.DataFrame(evoked_feature_rows)

    # --- 6. Calcul des features (Epochs) ---
    print("  ... 5/6 Extraction des features (Epochs)...")
    n_trials = len(epochs)
    code_to_label = {code: label for label, code in epochs.event_id.items()}
    conditions = [code_to_label.get(code, str(code)) for code in epochs.events[:, 2]]
    info = {'trial_index': np.arange(n_trials), 'condition': conditions, 'subject': [subject] * n_trials}
    
    df_epoch_features = pd.DataFrame(info)
    for config in FREQUENCY_BAND_WINDOWS:
        power = compute_band_power_epochs(
            epochs, CHANNEL_CLUSTERS.get(config['cluster'], []),
            config['freq_range'], config['tmin'], config['tmax'],
            n_freqs=config.get('n_freqs', 5),
            min_cycles=config.get('min_cycles', 3.0),
            max_cycles=config.get('max_cycles', 7.0),
            baseline=config.get('baseline', TFR_BASELINE),
            mode=config.get('mode', TFR_MODE),
            decim=config.get('decim', TFR_DECIM),
        )
        df_epoch_features[config['name']] = power
    
    print(f"  ... 6/6 Sujet {subject} terminé. {len(df_epoch_features)} essais extraits.")

    return df_evoked_features, df_epoch_features

# =============================================================================
# PARTIE 3 : BOUCLE D'EXÉCUTION PRINCIPALE (Corrigée)
# =============================================================================

# --- 1. Définir les Constantes de l'Analyse ---
subjects_list = ['01', '02', '03', '04', '05', '06', '07', '08', '10', '11', '13', '14']
session = '001'
run = '01'

# --- 2. Définir les Configurations de Features ---
CHANNEL_CLUSTERS = {
    "midline_fc": ["Fz", "Cz"],
    "centro_parietal": ["Cz", "Pz", "P3", "P4"],
    "parietal": ["Pz", "P3", "P4"],
    "occipital": ["O1", "O2"],
}
FREQUENCY_BAND_WINDOWS = [
    {"name": "theta_midline_fc", "cluster": "midline_fc", "freq_range": (5.0, 7.0), "n_freqs": 3, "tmin": 0.200, "tmax": 0.600},
    {"name": "alpha_parietal", "cluster": "parietal", "freq_range": (8.0, 13.0), "n_freqs": 6, "tmin": 0.200, "tmax": 0.600},
    {"name": "beta_parietal", "cluster": "parietal", "freq_range": (13.0, 20.0), "n_freqs": 8, "tmin": 0.200, "tmax": 0.600},
]

configs = {
    "CHANNEL_CLUSTERS": CHANNEL_CLUSTERS,
    "FREQUENCY_BAND_WINDOWS": FREQUENCY_BAND_WINDOWS
}

t_epoch = {'tmin': -0.2, 'tmax': 0.8, 'baseline': (-0.2, 0.0)}

# --- 3. Lancer la Boucle ---
all_evoked_features = []
all_epoch_features = []

print(f"Lancement du pipeline TFR pour {len(subjects_list)} sujets...")

for subject in subjects_list:
    df_evoked, df_epoch = process_subject_features(
            subject,
            session,
            run,
            configs=configs,
            t_epoch=t_epoch
        )
    all_evoked_features.append(df_evoked)
    all_epoch_features.append(df_epoch)
            
# --- 4. Combiner les Résultats ---
df_all_evoked_features = pd.concat(all_evoked_features, ignore_index=True)
df_all_epoch_features = pd.concat(all_epoch_features, ignore_index=True)

# --- 5. Sauvegarder les Résultats ---
output_epoch_csv = deriv_analysis / "all_subjects_epoch_band_features.csv"
output_evoked_csv = deriv_analysis / "all_subjects_evoked_band_features.csv"

df_all_evoked_features.to_csv(output_evoked_csv, index=False)
df_all_epoch_features.to_csv(output_epoch_csv, index=False)

print("\n--- Pipeline TFR Terminé ---")
print(f"DataFrame (Epochs) final : {df_all_epoch_features.shape[0]} essais x {df_all_epoch_features.shape[1]} features")
print(df_all_epoch_features.head())
print(f"\nDataFrame (Evoked) final : {df_all_evoked_features.shape[0]} conditions x {df_all_evoked_features.shape[1]} features")
print(df_all_evoked_features.head())
print(f"\nRésultats sauvegardés dans : {deriv_analysis}")

## 6. Synthèse et prochaines étapes

1.  L'exploration TFR (Temps-Fréquence) a illustré les différences Inconcurgent vs Concurgent, notamment dans les bandes thêta et alpha sur la ligne médiane.
2.  Vous avez factorisé le calcul de la puissance en fonctions robustes (ex: `compute_band_power_epochs`) qui gèrent les essais (`Epochs`) et les moyennes (`Evoked`).
3.  Le pipeline final génère et sauvegarde **trois DataFrames** consolidés pour *tous les sujets* :
    * `all_subjects_epoch_band_features.csv` : La matrice d'attributs pour le **Machine Learning** (1 ligne par essai).
    * `all_subjects_evoked_band_features.csv` : La matrice d'attributs pour l'**analyse statistique** (1 ligne par condition par sujet).
4.  Le prochain notebook (Module 8) pourra entraîner des modèles de classification directement sur le fichier `epoch_features.csv`, en le fusionnant éventuellement avec les features ERP que nous avons déjà extraites.